# Planejamento e validação das métricas — Aulas 22 e 23

**Atualização autorizada pelo aluno:** as barras de contagem foram mantidas; o gráfico por década foi substituído pela cobertura IMDb por categoria, e as duas barras de mediana foram substituídas por um boxplot de notas por tipo. As escolhas foram propostas com apoio de IA e adotadas após autorização. A justificativa pessoal e a reflexão individual exigidas pela atividade continuam pendentes; este registro técnico não as substitui.

Público: analistas de curadoria que exploram a base histórica da Netflix.
Pergunta: como a composição do recorte, a cobertura das notas e sua distribuição ajudam a comparar filmes e séries?

Filtros: tipo de conteúdo, intervalo numérico e inclusão explícita de ausências. Na aba de indicadores, escolher gênero, país ou tipo para agrupar e clicar nas barras de contagem para refinar os quatro cartões e os dois gráficos de detalhe.

Indicadores: total de títulos; cobertura IMDb; mediana IMDb; participação de séries.
Visuais: títulos por categoria; cobertura IMDb por categoria; boxplot das notas IMDb por tipo de conteúdo.

Fonte utilizada: o CSV tratado em `data/clean/dataset.csv`, derivado do dataset `netflix_content_intelligence_combined.csv` fornecido pelo aluno e de sua EDA. A base contém lançamentos até 2021; não descreve o catálogo atual. A fonte pública original e a licença continuam pendentes de identificação.

Ausências não são preenchidas, e os dados originais não são alterados pelos gráficos. No Dashboard 1, a visualização logarítmica de votos e popularidade fica ativada inicialmente e pode ser desligada.

In [1]:
# [IA] ChatGPT/Codex – carregar o CSV tratado para conferir os cálculos.
from pathlib import Path
import sys
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ))
df = pd.read_csv(RAIZ / 'data/clean/dataset.csv')

## Cálculo direto das métricas

Primeiro, calculamos cada indicador diretamente com pandas:

- **Total:** quantidade de linhas, considerando um título por registro.
- **Cobertura IMDb:** títulos com nota IMDb divididos pelo total.
- **Mediana IMDb:** valor central das notas disponíveis; valores ausentes ficam fora desse cálculo.
- **Participação de séries:** títulos do tipo Série divididos pelo total.

As proporções ficam entre 0 e 1; no aplicativo são exibidas como porcentagens. No recorte vazio, o total é 0 e os outros indicadores ficam indisponíveis. Se houver títulos, mas nenhum tiver nota, a cobertura é 0% e a mediana fica indisponível.

In [2]:
# [IA] ChatGPT/Codex – calcular diretamente os quatro indicadores com pandas.
total = len(df)
avaliados = int(df['imdb_rating'].notna().sum())
series = int((df['tipo_conteudo'] == 'Série').sum())

cobertura = avaliados / total if total > 0 else None
mediana = df['imdb_rating'].median() if avaliados > 0 else None
proporcao_series = series / total if total > 0 else None

metricas_notebook = {
    'total': total,
    'avaliados': avaliados,
    'series': series,
    'cobertura': cobertura,
    'mediana': mediana,
    'proporcao_series': proporcao_series,
}
pd.Series(metricas_notebook, name='Base completa')

total               7786.000000
avaliados           5460.000000
series              2410.000000
cobertura              0.701259
mediana                6.500000
proporcao_series       0.309530
Name: Base completa, dtype: float64

## Cálculo direto da cobertura por categoria

Para cada gênero principal, contamos todos os títulos com `size` e somente as notas disponíveis com `count`. A cobertura é a razão entre avaliados e total, mantida entre 0 e 1 e apresentada como porcentagem no gráfico.

Categorias ausentes recebem o rótulo “Não informado”, sem remover seus títulos. As dez categorias mais frequentes são ordenadas por total e, em caso de empate, pelo nome. Uma categoria com títulos, mas sem notas, tem cobertura 0%.

In [3]:
# [IA] ChatGPT/Codex – calcular a cobertura por categoria diretamente com pandas.
agrupamento = 'primary_genre'
dados_categorias = df.assign(categoria=df[agrupamento].fillna('Não informado'))
cobertura_categorias = (
    dados_categorias.groupby('categoria')['imdb_rating']
    .agg(total='size', avaliados='count')
    .reset_index()
)
cobertura_categorias['cobertura'] = (
    cobertura_categorias['avaliados'] / cobertura_categorias['total']
)
cobertura_categorias = (
    cobertura_categorias.sort_values(['total', 'categoria'], ascending=[False, True])
    .head(10)
    .reset_index(drop=True)
)
cobertura_categorias

,categoria,total,avaliados,cobertura
0,Dramas,1383,1185,0.856833
1,Comedies,1074,890,0.828678
2,Documentary,945,643,0.680423
3,Action & Adventure,721,621,0.861304
4,International TV Shows,690,470,0.681159
5,Children & Family Movies,502,267,0.531873
6,Crime TV Shows,369,246,0.666667
7,Kids,359,196,0.545961
8,Stand-Up Comedy,321,10,0.031153
9,Horror Movies,244,215,0.881148


## Conferência das notas por tipo de conteúdo

O boxplot utiliza as notas individuais disponíveis de filmes e séries, e apresenta sua distribuição e a mediana. O resumo abaixo permite conferir o total, a quantidade avaliada, a mediana e a cobertura de cada tipo.

`size` conta todos os títulos do grupo; `count` conta apenas notas disponíveis; `median` calcula a mediana dessas notas. Um grupo sem notas continua no resumo com cobertura 0 e mediana ausente. Não inventamos uma nota zero para esse grupo.

In [4]:
# [IA] ChatGPT/Codex – agrupar as notas por tipo e calcular a cobertura.
resumo_por_tipo = (
    df.groupby('tipo_conteudo')['imdb_rating']
    .agg(total='size', avaliados='count', mediana='median')
    .reset_index()
)
resumo_por_tipo['cobertura'] = resumo_por_tipo['avaliados'] / resumo_por_tipo['total']
resumo_por_tipo

,tipo_conteudo,total,avaliados,mediana,cobertura
0,Filme,5376,3965,6.3,0.737537
1,Série,2410,1495,7.1,0.620332


## Conferência com o aplicativo

Comparamos os cálculos diretos com as funções utilizadas pelo aplicativo. Conferimos os valores conhecidos da base, a seleção de Dramas, recortes vazios, títulos sem nota e uma categoria ausente em um pequeno exemplo. As verificações com `assert` interrompem a célula se algum resultado não corresponder ao esperado.

In [5]:
# [IA] ChatGPT/Codex – comparar os cálculos e verificar recortes vazios, categorias ausentes e falta de notas.
from src.analytics import (
    calculate_metrics, category_counts, category_coverage,
    select_categories, rating_summary,
)

m = calculate_metrics(df)
for nome, valor in metricas_notebook.items():
    assert m[nome] == valor, f'Resultado diferente para {nome}'

assert df['content_id'].is_unique
assert total == 7786
assert avaliados == 5460
assert series == 2410
assert cobertura == 5460 / 7786
assert mediana == 6.5
assert proporcao_series == 2410 / 7786

pd.testing.assert_frame_equal(category_coverage(df, agrupamento), cobertura_categorias)
pd.testing.assert_frame_equal(rating_summary(df), resumo_por_tipo)

vazio = calculate_metrics(df.iloc[:0])
assert vazio['total'] == 0
assert vazio['cobertura'] is None
assert vazio['mediana'] is None
assert vazio['proporcao_series'] is None
assert category_coverage(df.iloc[:0], agrupamento).empty
assert rating_summary(df.iloc[:0]).empty

titulos_sem_notas = df[df['imdb_rating'].isna()]
sem_notas = calculate_metrics(titulos_sem_notas)
assert sem_notas['cobertura'] == 0
assert sem_notas['mediana'] is None
assert (category_coverage(titulos_sem_notas, agrupamento)['cobertura'] == 0).all()
resumo_sem_notas = rating_summary(titulos_sem_notas)
assert (resumo_sem_notas['avaliados'] == 0).all()
assert resumo_sem_notas['mediana'].isna().all()

exemplo = pd.DataFrame({
    'primary_genre': ['Dramas', 'Dramas', None, 'Comedies'],
    'imdb_rating': [8.0, None, 7.0, None],
})
conferencia = category_coverage(exemplo, 'primary_genre').set_index('categoria')
assert conferencia.loc['Dramas', 'total'] == 2
assert conferencia.loc['Dramas', 'avaliados'] == 1
assert conferencia.loc['Dramas', 'cobertura'] == 0.5
assert conferencia.loc['Não informado', 'total'] == 1
assert conferencia.loc['Não informado', 'cobertura'] == 1
assert conferencia.loc['Comedies', 'avaliados'] == 0
assert conferencia.loc['Comedies', 'cobertura'] == 0

dramas = select_categories(df, 'primary_genre', [{'categoria': 'Dramas'}])
assert calculate_metrics(dramas)['total'] == int((df['primary_genre'] == 'Dramas').sum())
assert category_coverage(dramas, 'primary_genre')['total'].sum() == len(dramas)

print('Conferências concluídas: indicadores, cobertura, notas por tipo, vazios, ausências e seleção.')

Conferências concluídas: indicadores, cobertura, notas por tipo, vazios, ausências e seleção.


## Três visuais do Dashboard 2

1. **Títulos por categoria:** mostra a frequência das dez categorias mais comuns e permite selecionar categorias no aplicativo. A seleção refina os quatro indicadores, a cobertura e o boxplot.
2. **Cobertura IMDb por categoria:** mostra a porcentagem de títulos com nota nas categorias do recorte selecionado. A quantidade avaliada e o total acompanham o percentual.
3. **Boxplot de notas IMDb por tipo:** mostra a distribuição das notas disponíveis de filmes e séries, incluindo sua mediana. Títulos sem nota ficam fora desse gráfico e permanecem no denominador da cobertura.

No aplicativo, as barras de contagem mantêm o contexto dos filtros gerais. Sem seleção nas barras, indicadores e detalhes consideram todo esse recorte. A cobertura também exibe no máximo dez categorias, ordenadas por tamanho do grupo; a comparação não abrange automaticamente todas as categorias da base.

Abaixo usamos a base completa para conferir o formato e os dados dos três gráficos. A autorização para atualizar a implementação não substitui a justificativa individual do aluno.

In [6]:
# [IA] ChatGPT/Codex – preparar as barras de contagem por categoria.
from src.charts import category_bar, coverage_bar, rating_boxplot

category_bar(category_counts(df, agrupamento))

alt.Chart(...)

In [7]:
# [IA] ChatGPT/Codex – preparar o gráfico de cobertura IMDb por categoria.
coverage_bar(category_coverage(df, agrupamento))

alt.LayerChart(...)

In [8]:
# [IA] ChatGPT/Codex – preparar o boxplot com as notas individuais disponíveis.
rating_boxplot(df)

alt.Chart(...)

## Limitações

A cobertura de notas difere entre filmes e séries e também entre categorias. O boxplot representa apenas títulos avaliados; a comparação não mede causalidade, qualidade objetiva ou retorno financeiro. Gênero e país são os atributos principais já presentes no CSV, sem desdobrar títulos em múltiplas linhas.

A barra de cobertura de uma categoria sem avaliações tem valor 0%; isso descreve ausência de informação, e não uma nota 0. As categorias sem nome são mantidas como “Não informado”. Categorias fora das dez maiores não aparecem automaticamente nas barras, embora possam integrar os indicadores sem seleção.

Os dados são históricos e vão até 2021. Ausências e valores extremos não foram removidos da base nesta atualização.

Uso de IA: ChatGPT/Codex apoiou os cálculos, as verificações e a atualização dos gráficos após autorização do aluno. A justificativa pessoal das escolhas e a reflexão individual ainda devem ser redigidas pelo aluno; não foram preenchidas por este notebook.